# Fine-Resolution Graph Creation from GeoPackage

This notebook creates a high-resolution maritime navigation graph (0.02-0.3 NM) for detailed coastal and harbor routing. It uses progressive refinement techniques with regular grids or H3 hexagonal tessellation.

#### Workflow Overview

1. **Load Base Route** - Import existing route to define area of interest
2. **Buffer Route** - Create expanded zone around route for graph creation
3. **Filter ENCs** - Find charts covering the buffered area
4. **Generate Fine Grid** - Create high-resolution navigable polygon (0.1-0.3 NM or H3)
5. **Construct Graph** - Build dense NetworkX graph with connectivity analysis
6. **Calculate Route** - Compute optimal coastal path on fine-resolution graph

#### Data Flow

```
Base Route → Buffer → ENC Filtering → Fine Grid → Dense Graph → Coastal Route
```

#### Expected Outputs

- **Fine Graph**: High-resolution nodes (~180K at 0.1 NM, ~950K for H3)
- **Coastal Route**: Precise path following coastline geometry
- **Benchmarks**: Performance metrics for fine graph

#### Required Data

This notebook requires:
1. **Base Route**: Pre-computed route from base graph workflow (stored in GeoPackage)
2. **ENC Data**: S-57 charts converted to GeoPackage format
3. **Config File**: `src/nautical_graph_toolkit/data/graph_config.yml`

**Setup Instructions:** See `docs/SETUP.md`
**Troubleshooting:** See `docs/TROUBLESHOOTING.md`

## 1. Configuration

Centralized parameter configuration for the fine-resolution graph creation workflow.

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION
# =============================================================================
# This cell contains all configurable parameters for the notebook.
# Modify these settings to customize the graph creation workflow.

# --- General Workflow Flags ---
# Toggle major workflow steps on/off for testing or partial runs
buffer_sliced = True           # If True, slice buffer to reduce area (for testing/optimization)
graph_mode = "fine"            # Graph type: "fine" (regular grid) or "h3" (hexagonal tessellation)
save_gpkg = True              # Save graph to GeoPackage file
calc_route = True             # Calculate and visualize a route on the created graph

# --- Data Sources & Paths ---
data_file_name = "enc_west.gpkg"  # GeoPackage filename in output/ directory
base_route_name = "base_route_gpkg"
base_route_table = "base_routes"
config_file_path = "src/nautical_graph_toolkit/data/graph_config.yml"

# --- Geographic & Route Parameters ---
route_buffer_size_nm = 24.0           # Buffer distance around route in nautical miles
slice_south_degree = 37.0             # Southern latitude limit for buffer slicing
departure_port_name = "Pilot"         # Starting point name
arrival_port_name = "San Francisco"   # Ending point name

# --- Fine Graph Settings ---
# Parameters for regular grid graph creation
fine_grid_spacing_nm = 0.2            # Node spacing in nautical miles
fine_grid_max_points = 50000        # Safety limit to prevent excessive memory usage
fine_graph_max_edge_factor = 3.0      # Max edge length multiplier (also used for bridging)
fine_graph_bridge_components = True  # Bridge disconnected components (useful for spacing <0.1 NM)

# --- H3 Graph Settings ---
keep_largest_component = True         # Keep only the largest connected component

def _to_str(value_to_format: float) -> str:
    """
    Helper function to format a numeric value for file naming.

    It converts a float into a string representation of the value multiplied
    by 100, formatted to at least two digits with leading zeros. This is
    useful for creating consistent file suffixes from grid spacing values.

    Examples:
    - 0.1   -> "10"
    - 0.05  -> "05"
    - 1.0   -> "100"
    - 0.25  -> "25"

    Args:
        value_to_format (float): The numeric value to format.

    Returns:
        str: A string representation of the value * 100, zero-padded to at least 2 digits.
    """
    # 1. Multiply by 100 to shift the first two decimal places.
    #    Example: 0.05 -> 5.0; 0.1 -> 10.0; 1.0 -> 100.0
    value = value_to_format * 100
    # 2. Round to handle potential floating-point inaccuracies and convert to integer.
    #    Example: 5.0 -> 5; 10.0 -> 10; 100.0 -> 100
    int_value = int(round(value))
    # 3. Format the integer to a two-digit string with a leading zero if needed.
    #    Example: 5 -> "05"; 10 -> "10"; 100 -> "100"
    return f"{int_value:02d}"

# --- Output & Saving ---
gpkg_h3_name_suffix = "gpkg_6_11"
fine_grid_name_suffix = _to_str(fine_grid_spacing_nm)

# --- Configuration Summary ---
print("=" * 70)
print("✓ Configuration loaded successfully!")
print("=" * 70)
print(f"📍 Route: {departure_port_name} → {arrival_port_name}")
print(f"📊 Graph mode: {graph_mode}")
print(f"🛣️  Grid spacing: {fine_grid_spacing_nm} NM")
print(f"📐 Buffer size: {route_buffer_size_nm} NM")
if graph_mode == "h3":
    print(f"🔷 H3 mode: Multi-resolution hexagonal tessellation")
print("=" * 70)

### 1.1 Parameter Quick Reference

This notebook uses several parameters to control fine graph creation.

**graph_mode** - Graph type (default: fine)
- Options: "fine" (regular grid), "h3" (hexagonal)

**fine_grid_spacing_nm** - Node density (0.02-0.3 NM, default: 0.2)
- Lower = more detail, slower. 0.2 NM ≈ 180K nodes, 0.1 NM ≈ 720K nodes.

**route_buffer_size_nm** - Buffer around base route (5-50 NM, default: 24)
- Determines area for fine graph creation.

**fine_graph_bridge_components** - Bridge disconnected components (default: True)
- Recommended for spacing <0.1 NM to prevent routing failures.

**keep_largest_component** - Remove isolated clusters (default: True)
- Prevents routing failures across disconnected regions.

**For detailed explanations and tradeoffs, see APPENDIX below.**

### 1.2 Imports

This section sets up the Python environment and imports all necessary libraries for graph creation, geospatial processing, and visualization.

In [ ]:
import os
import sys
from pathlib import Path
import time
import pandas as pd
import plotly.express as px

import plotly.graph_objects as go
import plotly.io as pio
from dotenv import load_dotenv

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib

# Get project root for .env file loading
project_root = Path.cwd().parent.parent


from nautical_graph_toolkit.core.graph import (FineGraph, GraphConfigManager,
                                            H3Graph)
from nautical_graph_toolkit.core.pathfinding_lite import Route
from nautical_graph_toolkit.core.s57_data import (ENCDataFactory,
                                                S57AdvancedConfig)
from nautical_graph_toolkit.utils.geometry_utils import Buffer, Slicer
from nautical_graph_toolkit.utils.plot_utils import PlotlyChart
from nautical_graph_toolkit.utils.port_utils import Boundaries, PortData
from nautical_graph_toolkit.utils.notebook_utils import BenchmarkLogger, load_estimates

# Load environment variables from .env file at the project root
load_dotenv(project_root / ".env")
pio.renderers.default = "notebook_connected"

# Define paths for data and output
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)

# Define database file
data_file = project_root / "data" / data_file_name


print(f"Output directory: {output_dir}")
print(f"Data file: {data_file}")

# --- Performance Tracking ---
logger = BenchmarkLogger()
logger.configure_fine_graph(
    graph_mode=graph_mode,
    db_schema = data_file_name,
    spacing_nm=fine_grid_spacing_nm,
    max_points=fine_grid_max_points,
    buffer_size_nm=route_buffer_size_nm,
    buffer_sliced=buffer_sliced,
    keep_largest_component=keep_largest_component
)
logger.set_result('workflow', 'graph_fine_GPKG_v2')
logger.set_result('data_source', 'GeoPackage')
logger.set_result('bridge_components', fine_graph_bridge_components)

# --- Load Performance Estimates ---
estimate = load_estimates(
    notebook='graph_fine_GPKG_v2',
    graph_mode=graph_mode,
    spacing_nm=fine_grid_spacing_nm,
    backend='GeoPackage'
)

if estimate:
  print(f"⏱️  Estimated duration: {estimate['mean']:.1f} ± {estimate['std_dev']:.1f}s")
  print(f"   Based on {estimate['count']} previous runs")
  print(f"   Range: {estimate['min']:.1f}s - {estimate['max']:.1f}s")
else:
  print("⏱️  No historical benchmark data available")

### 1.3 Workflow Context

**Pipeline Position**: Step 2.5 of 3 (Advanced Graph Construction)
1. **Data Import** (`import_s57.ipynb`) - Convert S-57 ENCs to GeoPackage, PostGIS, or SpatiaLite backend
2. **Base Graph Construction** (`graph_GeoPackage_v2.ipynb`, `graph_PostGIS_v2.ipynb`, or `graph_SpatiaLite_v2.ipynb`)
3. **Fine-Resolution Graph** (This notebook) - Create high-detail graphs (regular grid or H3 hexagonal)
4. **Weighting & Routing** (`graph_weighted_directed_GeoPackage_v2.ipynb` or similar) - Add edge weights and optimize routes

**Prerequisites**:
- Completed base graph workflow (steps 1-2)
- Base route saved from `graph_GeoPackage_v2.ipynb`
- ENC data converted to GeoPackage/SpatiaLite format
- Configuration file: `src/nautical_graph_toolkit/data/graph_config.yml`

**Outputs**:
- High-resolution maritime navigation graph (40K-950K nodes depending on mode and spacing)
- Fine-resolution route with detailed waypoint geometry
- Performance benchmarks appended to `benchmark_graph_fine.csv`

**Graph Modes**:
- **Fine Grid** (regular square grid): 0.1-0.3 NM spacing, suitable for coastal and harbor routing
- **H3 Hexagonal** (hierarchical hexagons): Multiple resolutions (6-11), optimal for multi-scale routing

**Next Steps**:
- Use fine graph for precise harbor routing and detailed analysis
- Compare routing results between spacing values (0.3 vs 0.1 NM) to understand resolution-accuracy tradeoffs
- Proceed to `graph_weighted_directed_GeoPackage_v2` notebooks for optimized routing with edge weights

### 1.4 Initialize GeoPackage/SpatiaLite Data Factory

The ENCDataFactory provides a unified interface for accessing S-57 ENC data from GeoPackage. It handles file connections, layer queries, and spatial filtering.

In [ ]:
# --- Initialize ENC Data Factory for GeoPackage Backend ---
# The factory provides a unified interface for accessing ENC data
# from GeoPackage. It will be used throughout the notebook
# for querying navigational layers and saving results.
gpkg_factory = ENCDataFactory(source=data_file)

### 1.5 Initialize Plotly Visualization

Create the base Plotly map that will be reused throughout the notebook for visualizing routes, buffers, grids, and other geographic data.

In [ ]:
# --- Create Base Plotly Map ---
# Initialize the plotting utility and create an interactive base map
# The Mapbox token is loaded from .env for security (not hardcoded)
# The base map will be copied for different visualizations throughout the notebook
ply = PlotlyChart()
ply_fig = ply.create_base_map(mapbox_token=os.getenv('MAPBOX_TOKEN'))
ply.plotly_base_config(ply_fig)

## 2. Base Route Loading and Area of Interest Definition

Fine-resolution graphs are typically focused on specific routes to avoid creating unnecessarily large graphs. We start by loading an existing base route and creating a buffer around it to define our area of interest.

In [ ]:
# --- Load Pre-computed Base Route from GeoPackage file ---
# This route was created in the base graph workflow and stored in GeoPackage
# We'll use it to focus our fine-resolution graph creation on relevant areas
logger.start_timer('load_base_route')
base_route = gpkg_factory.load_route(route_name=base_route_name,
                                     table_name=base_route_table)

elapsed = logger.end_step('load_base_route')
print(f"Loading base route took: {elapsed:.2f}s")

# --- Visualize Base Route ---
# Create a deep copy of the base map to avoid modifying the original
# This allows us to create multiple independent map visualizations
ply_route = go.Figure(ply_fig)

ply.add_route_trace(figure=ply_route,
                    line=base_route,
                    name="Base Route")
ply_route.show()

### 2.1 Create Buffer Around Route

Create a buffer zone around the base route to define the area where we'll build our fine-resolution graph. The buffer size is configurable and typically set to 12-24 nautical miles.

In [ ]:
# --- Create Buffer Zone Around Base Route ---
# Buffer creates a polygon extending route_buffer_size_nm nautical miles
# on both sides of the route. This defines our area of interest for
# fine-resolution graph creation.
logger.start_timer('create_buffer')
route_buffer = Buffer.create_buffer(base_route, route_buffer_size_nm)
elapsed = logger.end_step('create_buffer')
print(f"Creating buffer took: {elapsed:.2f}s")

In [ ]:
# --- Visualize Buffer on Map ---
# Add the buffer polygon to the map to verify it covers the desired area
ply.add_polygon_trace(fig=ply_route,
                      polygon=route_buffer,
                      name="Base Route Buffer")
ply_route.show()

### 2.2 Slice Buffer (Optional Area Reduction)

For testing or optimization, we can slice the buffer to reduce the area of interest. This is useful when experimenting with different grid spacings or graph settings without processing the entire route buffer.

In [ ]:
# --- Slice Buffer to Reduce Area (Optional) ---
# Use geographic bounding box to clip the buffer polygon
# This reduces processing time and graph size for testing
# Set buffer_sliced=False in settings to use full buffer
logger.start_timer('slice_buffer')
sliced_buffer = Slicer.slice_by_bbox(route_buffer, south=slice_south_degree)
elapsed = logger.end_step('slice_buffer')
print(f"Slicing buffer took: {elapsed:.2f}s")

# --- Visualize Sliced Buffer ---
# Add the sliced buffer to the map to verify the reduced area
ply.add_polygon_trace(fig=ply_route,
                      polygon=sliced_buffer,
                      name="Sliced Buffer")
ply_route.show()

## 3. Graph Creation Prerequisites

Before creating the graph, we need to:
1. Define departure and arrival points (ports or custom coordinates)
2. Load graph configuration (navigable/obstacle layers, H3 settings)
3. Filter ENCs to only those intersecting our area of interest

### 3.1 Define Departure and Arrival Points

These points will be used for routing after the graph is created. They can come from the World Port Index or be custom user-defined coordinates.

#### Port Selection: World Port Index or Custom Points

The PortData utility allows you to:
- Query ports from the World Port Index by name
- Create custom port entries with specific coordinates
- Update existing custom ports with new coordinates

In [ ]:
# --- Define Geographic Points for Routing ---
# Initialize port data utility which merges World Port Index with custom ports.
# This allows using standard ports (e.g., "San Francisco") or defining custom
# test points (e.g., "Pilot") for route calculations.
port = PortData()

# Create or update a custom port entry.
# The 'if_exists' parameter controls behavior when a port already exists:
#   - 'update': Replace coordinates with new values
#   - 'skip': Keep existing port unchanged
#   - 'raise': Throw an error if port exists
port.create_custom_port(port_name=departure_port_name,
                        lon=-122.27,
                        lat=37.0,
                        if_exists='update')

# Get departure and arrival points by name.
# These will be used as endpoints for route calculation.
dep_point = port.get_port_by_name(departure_port_name)
arr_point = port.get_port_by_name(arrival_port_name)

# Create detailed DataFrames for inspection (optional)
port1_df = port.get_port_details_df(dep_point)
port2_df = port.get_port_details_df(arr_point)

# Print formatted port information
print(port.format_port_string(dep_point))
print(port.format_port_string(arr_point))

# --- Visualize Ports on Map ---
# Add departure port (blue) and arrival port (red) to the map
ply.add_single_port_trace(ply_route, dep_point, name=dep_point['PORT_NAME'], color='blue')
ply.add_single_port_trace(ply_route, arr_point, name=arr_point['PORT_NAME'], color='red')
ply_route.show()

### 3.2 Load Graph Configuration and Filter ENCs

The graph configuration YAML file defines which S-57 layers to use for navigable areas and obstacles. We also filter ENCs to only those intersecting our area of interest to optimize performance.

In [ ]:
# --- Select Active Buffer (Full or Sliced) ---
# Choose which buffer polygon to use based on the buffer_sliced setting.
# Slicing is useful for:
#   - Testing workflows on smaller areas (faster iteration)
#   - Reducing memory requirements for experimentation
#   - Focusing on specific geographic regions of interest
if buffer_sliced:
    active_buffer = sliced_buffer
else:
    active_buffer = route_buffer

# --- Load Graph Configuration from YAML ---
# The config file (graph_config.yml) defines:
# - Navigable layers: Define safe water areas for routing
#   * seaare: Main sea areas from ENC charts
#   * fairwy: Designated fairways and channels
#   * drgare: Dredged areas (maintained depth)
#   * tsslpt: Traffic separation scheme lanes
#   * prcare: Precautionary areas
# - Obstacle layers: Define hazards to subtract from navigable areas
#   * lndare: Land areas (hard obstacles)
#   * slcons: Shoreline constructions (piers, breakwaters)
#   * obstrn: Obstructions (underwater hazards)
# - H3 hexagon settings: Resolution ranges and connectivity rules
#   * Multi-resolution hierarchy (typically 6-11 for coastal navigation)
#   * Bridge connectivity parameters for seamless multi-scale routing
config_path = project_root / config_file_path
config_manager = GraphConfigManager(config_path)

# --- Filter ENCs by Boundary ---
# Query GeoPackage to find all ENCs that intersect our area of interest.
# This spatial filtering is CRITICAL for performance:
#   - Reduces data processing to relevant charts only
#   - Prevents loading unnecessary ENC data (thousands of charts)
#   - Typical workflow: 6000+ total ENCs → 20-50 relevant ENCs
# The method queries the enc_summary layer which contains bounding boxes
# for all available charts, using spatial filtering to identify intersections.
logger.start_timer('enc_filtering')
enc_list = gpkg_factory.get_encs_by_boundary(active_buffer)

elapsed = logger.end_step('enc_filtering')
print(f"ENC filtering took: {elapsed:.2f}s")

## 4. Fine Graph Creation (Regular Grid)

Fine graphs use regular rectangular grids with dense node spacing (0.1-0.3 NM) to provide high-resolution routing. This approach is ideal for:
- Detailed coastal navigation
- Harbor and channel routing
- Areas requiring precise path planning

The process involves:
1. Creating a navigable grid from S-57 layers (similar to base graph but with finer resolution)
2. Generating a dense node network within the navigable area
3. Connecting adjacent nodes with edges
4. Filtering to keep only the largest connected component

### 4.1 Fine Grid Creation

Create a navigable grid by querying S-57 layers and combining them into a single polygon. The FineGraph class uses an iterative approach, processing usage bands in priority order to build up coverage progressively.

In [ ]:
# --- Extract Layer Configuration from YAML ---
# Get the layer settings from the config file
layers_config = config_manager.get_value("layers")

# Extract the specific lists for navigable and obstacle layers.
# These layer definitions follow IHO S-57 standard object classes:
#
# Navigable layers define safe water areas (UNION operation):
#   - seaare: General sea areas (primary coverage)
#   - fairwy: Designated fairways with maintained depth
#   - drgare: Dredged areas with known depth
#   - tsslpt: Traffic separation lanes (routing corridors)
#   - prcare: Precautionary areas (navigable but requiring caution)
#
# Obstacle layers define hazards (SUBTRACTION operation):
#   - lndare: Land areas (hard obstacles)
#   - slcons: Shoreline constructions (piers, jetties, breakwaters)
#   - obstrn: Underwater obstructions (wrecks, rocks, etc.)
#
# The final navigable grid = (UNION of navigable) - (UNION of obstacles)
navigable_layers_config = layers_config.get('navigable', [])
obstacle_layers_config = layers_config.get('obstacles', [])

# --- Create Fine Grid ---
# Skip this section if graph_mode is set to "h3"
logger.start_timer('fine_grid_creation')

if graph_mode == "fine" or "h3":
    # Fine grid is not used for H# Graph creation but will be used in static land filtering. As it creates refined land and sea grids from all Usage band levels.
    # Initialize FineGraph class with GeoPackage data factory
    fg = FineGraph(data_factory=gpkg_factory,
                   route_schema_name="routes",
                   graph_schema_name="graph")

    # Create the navigable grid by processing S-57 layers iteratively.
    # The method uses a prioritized approach based on ENC usage bands:
    #   Band 1: Overview (1:1,500,000+) - Coarse ocean coverage
    #   Band 2: General (1:350,000-1:1,499,999) - Coastal approaches
    #   Band 3: Coastal (1:90,000-1:349,999) - Near-shore navigation
    #   Band 4: Approach (1:30,000-1:89,999) - Harbor approaches
    #   Band 5: Harbour (1:12,000-1:29,999) - Port areas
    #   Band 6: Berthing (<1:11,999) - Detailed harbor/dock areas
    #
    # Processing order: 1→6 (coarse to fine) to progressively refine coverage.
    # Each band's features are merged with existing grid, with higher priority
    # bands overwriting lower priority areas where they overlap.
    fg_grid = fg.create_fine_grid(route_buffer=active_buffer,
                                  enc_names=enc_list,
                                  navigable_layers=navigable_layers_config,
                                  obstacle_layers=obstacle_layers_config,
                                  return_geometries=True  # Ensure Shapely geometries are returned
                                  )
    elapsed = logger.end_step('fine_grid_creation')
    print(f"Fine grid creation took: {elapsed:.2f}s")

### 4.2 Visualize Fine Grid Components

Plot the different components of the generated grid to verify coverage. The combined_grid (red) represents the final navigable area after merging all layers.

In [ ]:
# --- Visualize Grid Components on Map ---
# Display the grid components to verify coverage:
# - main_grid (blue): Primary sea area from 'seaare' layer
# - combined_grid (red): Final merged navigable polygon used for graph creation
ply_fine_grid = go.Figure(ply_fig)
if graph_mode == "fine":
    ply.add_grid_trace(ply_fine_grid, name= "Main Grid", grid_geojson=fg_grid["main_grid"], color="blue")
    ply.add_grid_trace(ply_fine_grid, name= "Combined Grid", grid_geojson=fg_grid["combined_grid"], color="red")
    ply.add_grid_trace(ply_fine_grid, name= "Land Grid", grid_geojson=fg_grid["land_grid"], color="yellow")
    ply_fine_grid.show()

### 4.3 Fine Graph Construction

Convert the navigable grid polygon into a NetworkX graph by populating it with a dense network of nodes and edges. The spacing parameter controls node density (smaller = more nodes = longer processing time).

**Component Bridging for Fine Grids (<0.1 NM):**

When using very fine spacing (<0.1 NM), you may encounter artificial gaps between components due to:
- Numerical precision limits
- Slight polygon boundary misalignments
- Grid generation artifacts

The `bridge_components` parameter addresses this by:
1. Identifying disconnected components in the graph
2. Finding boundary nodes (nodes with fewer than 8 neighbors)
3. Adding bridge edges between nearby components within `max_edge_factor * spacing` distance
4. Prioritizing closest connections to maintain graph quality

**Usage:** Set `fine_graph_bridge_components=True` in settings when using spacing <0.1 NM.

In [ ]:
# --- Create Fine-Resolution Graph from Grid ---
# This is the most computationally intensive step. It creates a dense node
# network within the navigable polygon and connects adjacent nodes.
#
#
# Parameters:
# - spacing_nm: Distance between adjacent nodes in nautical miles
# - max_points: Safety limit to prevent excessive memory usage
# - max_edge_factor: Max edge length = spacing * max_edge_factor (also used for bridging)
# - bridge_components: If True, bridges disconnected components (recommended for spacing <0.1 NM)
# - keep_largest_component: Remove isolated node clusters for routing reliability
logger.start_timer('fine_graph_creation')
if graph_mode == "fine":
    G_fine = fg.create_base_graph(grid_data=fg_grid["combined_grid"],
                         spacing_nm=fine_grid_spacing_nm,
                         max_points=fine_grid_max_points,
                         max_edge_factor=fine_graph_max_edge_factor,
                         bridge_components=fine_graph_bridge_components,
                         keep_largest_component=keep_largest_component)
    elapsed = logger.end_step('fine_graph_creation')
    logger.set_result('node_count', G_fine.number_of_nodes())
    logger.set_result('edge_count', G_fine.number_of_edges())
    print(f"Graph has {G_fine.number_of_nodes():,} nodes and {G_fine.number_of_edges():,} edges")
    print(f"Fine graph creation took: {elapsed:.2f}s")

## 5. H3 Graph Creation (Hexagonal Tessellation)

H3 graphs use Uber's H3 hexagonal tessellation system instead of regular square grids. Benefits include:
- **Uniform neighbor distances**: All adjacent hexagons are equidistant
- **Better angular coverage**: 6 neighbors vs 8 in square grids (no diagonal artifacts)
- **Hierarchical resolution**: H3's multi-resolution structure allows mixed detail levels
- **Efficient connectivity**: Natural bridging between resolution levels

The H3Graph class handles multi-resolution hexagon creation, obstacle subtraction, and connectivity between resolution levels.

### 5.1 H3 Grid and Graph Creation

Create an H3-based graph in a single operation. The method handles grid creation, obstacle subtraction, graph construction, and connectivity bridging automatically.

In [ ]:
# --- Create H3 Hexagonal Graph ---
# This section only runs if graph_mode is set to "h3"
#
# H3 Graph Advantages:
# -------------------------
# 1. Uniform neighbor distances: All adjacent hexagons are equidistant
#    (eliminates diagonal distance artifacts from square grids)
# 2. Better angular coverage: 6 neighbors vs 8 in square grids
# 3. Hierarchical resolution: Natural multi-scale structure
#    - Parent hexagons subdivide into 7 child hexagons
#    - Seamless bridging between resolution levels
# 4. Efficient connectivity: Natural connectivity across resolution scales
#
# Performance benchmarks (Buffer=24NM, Resolutions 6-11, GeoPackage):
# - H3 generation: ~1m 21s (947,961 hexagons)
# - Graph construction: ~1m 24s (2,832,202 edges)
# - Component selection: ~45s (945,918 final nodes)
# - Total: ~3m 30s
#
# The H3 graph creation process:
# 1. Generates hexagons at multiple resolutions within navigable areas
#    - Each resolution level provides different detail (coarse to fine)
#    - Resolution 6: ~36 km² per hexagon (ocean navigation)
#    - Resolution 11: ~4 m² per hexagon (harbor precision)
# 2. Subtracts obstacles from hexagon coverage (land, constructions, obstructions)
# 3. Creates graph edges between adjacent hexagons (same resolution)
# 4. Bridges between resolution levels for seamless navigation
#    - Connects coarse and fine areas without artificial boundaries
#    - Enables smooth transitions from ocean to harbor routing
# 5. Optionally keeps only the largest connected component
#    - Removes isolated nodes/islands for routing reliability

logger.start_timer('h3_graph_creation')
if graph_mode == "h3":
    # Initialize H3Graph class with GeoPackage data factory
    h3 = H3Graph(data_factory=gpkg_factory,
                route_schema_name="routes",
                graph_schema_name="graph")

    # Load H3 settings from configuration YAML.
    # Includes resolution ranges, bridge settings, and connectivity rules:
    #   - resolution_ranges: Dict mapping usage bands to H3 resolution levels
    #   - connectivity.bridge_between_resolutions: Enable multi-scale bridging
    #   - connectivity.min_same_res_neighbors: Minimum same-res neighbors before bridging
    #   - connectivity.target_total_neighbors: Target total neighbor count with bridges
    #   - connectivity.max_bridge_distance_nm: Max distance for bridge connections
    h3_settings = config_manager.get_value("h3_settings")
    connectivity_config = h3_settings.get('connectivity', {})

    # Create H3 graph and grid in one operation.
    # Returns both the NetworkX graph and the hexagon GeoDataFrame for visualization.
    # The graph includes:
    #   - Node attributes: lon, lat, h3_index, resolution, geometry
    #   - Edge attributes: length (nautical miles), bridge (boolean flag)
    G_h3, h3_grid = h3.create_h3_graph(route_buffer=active_buffer,
                                       enc_names=enc_list,
                                       navigable_layers=navigable_layers_config,
                                       obstacle_layers=obstacle_layers_config,
                                       connectivity_config=connectivity_config,
                                       keep_largest_component=keep_largest_component)

    elapsed = logger.end_step('h3_graph_creation')
    logger.set_result('node_count', G_h3.number_of_nodes())
    logger.set_result('edge_count', G_h3.number_of_edges())
    print(f"Graph has {G_h3.number_of_nodes():,} nodes and {G_h3.number_of_edges():,} edges")
    print(f"H3 graph creation took: {elapsed:.2f}s")

## 6. Save Graph to GeoPackage

GeoPackage format is ideal for:
- Offline analysis and visualization (QGIS, ArcGIS)
- Sharing graphs with collaborators
- Archiving graph snapshots
- Fast save/load operations (no database overhead)

### 6.1 Save to GeoPackage (Fastest, Most Portable)

GeoPackage format is ideal for:
- Offline analysis and visualization (QGIS, ArcGIS)
- Sharing graphs with collaborators
- Archiving graph snapshots
- Fast save/load operations (no database overhead)

In [ ]:
# --- Save Graph to GeoPackage File ---
# GeoPackage (.gpkg) is a SQLite-based format optimized for geospatial data.
# It creates separate layers for nodes and edges with spatial indexes.
#
# GeoPackage Advantages:
# ----------------------
# 1. Portability: Single-file format, easy to share and archive
# 2. No server required: Works offline without database setup
# 3. Open standard: Supported by QGIS, ArcGIS, and most GIS tools
# 4. Fast read/write: SQLite backend with spatial indexing
# 5. Cross-platform: Works on Windows, Mac, Linux without modification
#
# Performance benchmarks (GeoPackage backend):
# - Fine graph (46K nodes, 181K edges): ~4s
# - H3 graph (945K nodes, 2.8M edges): ~2m
#
# The saved GeoPackage contains two layers:
#   - nodes: Point geometries with attributes (node_id, lon, lat, h3_index, resolution)
#   - edges: LineString geometries with attributes (source, target, length, bridge)
#
# Use cases:
#   - Opening in QGIS for visualization and analysis
#   - Sharing graphs with collaborators (no database setup required)
#   - Archiving graph snapshots for version control
#   - Loading back into Python with gpd.read_file() for further processing

logger.start_timer('save_gpkg')
if save_gpkg:
    if graph_mode == "h3":
        name = f"{graph_mode}_graph_{gpkg_h3_name_suffix}.gpkg"
        h3.save_graph_to_gpkg(G_h3, output_path=output_dir/ name)
        h3.save_grid_to_gpkg(fg_grid["land_grid_geom"], layer_name="land_grid", output_path=output_dir / name)
        h3.save_grid_to_gpkg(fg_grid["combined_grid_geom"], layer_name="sea_grid", output_path=output_dir / name)
    else:
        name = f"{graph_mode}_graph_{fine_grid_name_suffix}.gpkg"
        fg.save_graph_to_gpkg(G_fine, output_path=output_dir / name)
        fg.save_grid_to_gpkg(fg_grid["land_grid_geom"], layer_name="land_grid", output_path=output_dir / name)
        fg.save_grid_to_gpkg(fg_grid["combined_grid_geom"], layer_name="sea_grid", output_path=output_dir / name)
    elapsed = logger.end_step('save_gpkg')
    print(f"Saving to GeoPackage took: {elapsed:.2f}s")

## 7. Route Calculation on Fine Graph

With the fine-resolution graph created, we can now compute optimal routes with higher precision than the base graph. The finer grid spacing provides:
- More accurate coastline following
- Better channel navigation
- Improved route optimization in complex areas

### 7.1 Compute Route Using A* Algorithm

The Route class uses the A* pathfinding algorithm to find the shortest path between departure and arrival points. The algorithm considers only distance in this base route calculation (weights can be added later).

In [ ]:
# --- Calculate Route on Fine-Resolution Graph ---
# Use the A* algorithm to find the shortest path between departure and arrival points.
#
# A* Algorithm Benefits:
# ----------------------
# 1. Optimal pathfinding: Guaranteed to find shortest path (with admissible heuristic)
# 2. Efficient search: Uses heuristic to guide exploration toward goal
# 3. Graph-based: Works on any connected graph structure (fine grid or H3)
#
# The routing process:
# 1. Maps user-provided port coordinates to nearest graph nodes
# 2. Applies A* with Euclidean distance heuristic
# 3. Returns route geometry (LineString) and total distance (nautical miles)
#
# Current implementation uses base distance weighting (all edges weighted equally
# by length). Future enhancements can add:
#   - Traffic separation scheme priorities
#   - Depth-based routing (avoid shallow areas)
#   - Weather/current integration
#   - Vessel-specific constraints (draft, size)

logger.start_timer('route_calculation')
if calc_route:
    # Get the departure and arrival port geometries
    dep_point = port.get_port_by_name(departure_port_name)
    arr_point = port.get_port_by_name(arrival_port_name)

    # Select the appropriate graph based on mode
    if graph_mode == 'h3':
        graph_for_routing = G_h3
    else:
        graph_for_routing = G_fine

    # Initialize Route class with the graph and data manager.
    # The data manager provides GeoPackage connectivity for saving/loading routes.
    route = Route(graph=graph_for_routing, data_manager=gpkg_factory.manager)
    
    # Compute the route using A* algorithm.
    # The method automatically:
    #   1. Maps port coordinates to nearest graph nodes using spatial index
    #   2. Validates start/end nodes are in the graph
    #   3. Runs A* pathfinding with distance-based weighting
    #   4. Converts node sequence to route geometry (LineString)
    #   5. Calculates total distance in nautical miles
    result = route.base_route(
        departure_point=dep_point.geometry,
        arrival_point=arr_point.geometry,
    )

    if result is None:
        print("ERROR: No route found!")
    # Add diagnostics here
    else:
        route_geometry, distance = result

    elapsed = logger.end_step('route_calculation')
    print(f"Route calculation took: {elapsed:.2f}s")

### 7.1 Visualize Computed Route

Display the calculated route on an interactive map to verify the path makes sense and follows navigable channels appropriately.

In [ ]:
# --- Visualize Route on Interactive Map ---
# Create a new map figure and add the route, departure port, and arrival port
if calc_route:
    ply_fine_route = go.Figure(ply_fig)
    # Add the computed route as a line
    ply.add_route_trace(figure=ply_fine_route,
                        line=route_geometry,
                        name="Base Route")
    # Add departure port marker (blue)
    ply.add_single_port_trace(ply_fine_route, dep_point, name=dep_point['PORT_NAME'], color='blue')
    # Add arrival port marker (red)
    ply.add_single_port_trace(ply_fine_route, arr_point, name=arr_point['PORT_NAME'], color='red')
    ply_fine_route.show()


## Performance Summary and Benchmark Export

This section visualizes the time taken for each step of the pipeline and exports detailed performance metrics to CSV for long-term tracking and analysis.

In [ ]:
# --- Export Benchmark to CSV ---
csv_path = logger.export_benchmark()
print(f"\n💾 Benchmark saved to: {csv_path}")

# --- Display Benchmark Summary ---
print(logger.get_current_benchmark_summary())

In [ ]:
# --- Visualize Pipeline Performance ---
fig = logger.visualize_performance(
    title='Fine Graph Pipeline Performance (GeoPackage)',
    sort_by='time_descending',
    show=True
)

## APPENDIX: Detailed Parameter Documentation

### A.1 graph_mode - Graph Type Selection

**What it does:** Chooses between regular square grid or H3 hexagonal tessellation.

**Options:**
- **fine**: Regular grid (0.02-0.3 NM spacing)
- **h3**: Hierarchical hexagons (multi-resolution 6-11)

**Performance Comparison (Buffer=24NM, PostGIS):**
| Mode | Nodes | Total Time | Best For |
|------|-------|------------|----------|
| fine 0.2nm | 180K | 21 min | Production (RECOMMENDED) |
| h3 (6-11) | 894K | 107 min | Research/multi-resolution |

**How to Choose:**
- Ocean/coastal routing: fine 0.2 NM (balanced detail/speed)
- Harbor navigation: fine 0.1 NM (high detail)
- Multi-scale analysis: h3 (hierarchical coverage)

### A.2 fine_grid_spacing_nm - Node Density Control

**What it does:** Controls distance between adjacent nodes in nautical miles.

**Units:** Nautical miles between nodes

**Typical Values:**
- `0.3 NM`: Fast, lower detail (~40K nodes)
- `0.2 NM`: RECOMMENDED balance (~180K nodes)
- `0.1 NM`: High detail, slower (~720K nodes)

**Node Count Scaling:** Quadratic relationship (0.1 NM has 4× more nodes than 0.2 NM)

**Performance Impact (PostGIS, Buffer=24NM):**
| Spacing | Nodes | Graph Time | Total Time | Memory |
|---------|-------|------------|------------|--------|
| 0.3 NM | 40K | 5 min | 14 min | 4 GB |
| 0.2 NM | 180K | 12 min | 21 min | 8 GB |
| 0.1 NM | 720K | 45 min | 90 min | 16 GB |

**Component Bridging:**
- At <0.1 NM spacing, enable `fine_graph_bridge_components=True`
- Bridges disconnected components caused by numerical precision limits
- Adds edges between nearby boundary nodes

**How to Choose:**
```python
# Ocean crossing (fast)
fine_grid_spacing_nm = 0.3

# Coastal routing (RECOMMENDED)
fine_grid_spacing_nm = 0.2

# Harbor navigation (detailed)
fine_grid_spacing_nm = 0.1
```

### A.3 route_buffer_size_nm - Area Control

**What it does:** Expands area around base route for fine graph creation.

**Units:** Nautical miles

**Typical Values:**
- `12 NM`: Minimal, tight around route
- `24 NM`: Standard default (RECOMMENDED)
- `50 NM`: Large, includes alternative routes

**Impact:**
- Larger buffer = more area processed = slower execution
- Smaller buffer = faster but may miss routing alternatives

### A.4 fine_graph_bridge_components - Gap Bridging

**What it does:** Bridges disconnected components in fine graphs.

**When to Enable:**
- Spacing <0.1 NM (numerical precision issues)
- Irregular coastlines with complex geometry
- Multiple islands/channels in area

**Algorithm:** Finds boundary nodes (< 8 neighbors), connects nearest nodes across components within `max_edge_factor * spacing` distance.

**Performance:** Negligible impact (<1% total time)

### A.5 keep_largest_component - Connectivity Filtering

**What it does:** Removes isolated node clusters after graph creation.

**Values:**
- `True`: Keep only largest connected component (RECOMMENDED)
- `False`: Keep all nodes, including isolated clusters

**Impact:**
- True: Guarantees all routes connect (prevents failures)
- False: Maximum coverage but routing may fail

### A.6 Summary Table: Parameter Selection by Use Case

| Use Case | graph_mode | spacing_nm | buffer_size_nm | bridge_components |
|----------|-----------|------------|----------------|-------------------|
| Quick Prototyping | fine | 0.3 | 12 | False |
| Production Routing | fine | 0.2 | 24 | False |
| Harbor Navigation | fine | 0.1 | 12 | True |
| Multi-Resolution | h3 | N/A | 24 | N/A |
| Research/Analysis | h3 | N/A | 50 | N/A |

### A.7 Performance Formula and Estimation

**Rough Estimation:**
```
Total Time ≈ (Grid Time) + (Graph Time) + (Save Time) + (Route Time)

Where:
  Grid Time ≈ 2-5 min (ENC processing)
  Graph Time ≈ nodes ÷ 15,000 minutes (PostGIS)
  Save Time ≈ nodes ÷ 50,000 minutes (PostGIS optimized)
  Route Time ≈ 0.5-2 min (A* pathfinding)
```

**Example (PostGIS, 0.2 NM, Buffer=24NM):**
```
Grid: 3 min
Graph: 180K ÷ 15K = 12 min
Save: 180K ÷ 50K = 3.6 min
Route: 1 min
Total: ~20 min
```

**For measured benchmarks and hardware specifications, see:**
- `docs/TECHNICAL_SPECS.md` - Comprehensive performance data
- `output/benchmark_graph_fine.csv` - Historical runs

---

## APPENDIX B: Backend Information

This notebook uses **GeoPackage** as the data backend.

**Backend Quick Reference:**
| Backend | Best For | Performance vs GeoPackage |
|---------|----------|---------------------------|
| **PostGIS** | Production, 1000+ ENCs | 2.0-2.4× faster |
| **GeoPackage** | Portable, 100-1000 ENCs | Baseline (1.0×) |

**Switching Backends:**
- PostGIS version: `graph_fine_PostGIS_v2.ipynb`
- GeoPackage version: `graph_fine_GeoPackage_v2.ipynb` (this notebook)

**GeoPackage-Specific Notes:**
- Single-file format, no server required
- Portable, works offline
- R-Tree spatial indexing via pysqlite3
- Best for sharing/archiving results

**For detailed backend comparison, see:**
- `docs/SETUP.md` - Setup and feature comparison
- `docs/WORKFLOW_GEOPACKAGE_GUIDE.md` - Performance tradeoffs